# IFLS4–IFLS5: BLSM 2013 Python data audit

**Research question:** What is the ATT of receiving BLSM 2013 on real monthly household consumption per capita in IFLS5, relative to a comparable untreated outcome? We will later estimate subgroup-specific ATT by baseline characteristics.

This notebook **does not estimate a causal effect**. First verify transfer codes/timing, panel membership, expenditure definitions, and interview dates. No restricted microdata should be committed to GitHub.

In [ ]:
from pathlib import Path
import os
import pandas as pd

# RStudio project marker also identifies this project's Python analysis root.
root = Path.cwd()
if not (root / "IFLS_Project.Rproj").exists() and (root.parent / "IFLS_Project.Rproj").exists():
    root = root.parent
assert (root / "IFLS_Project.Rproj").is_file(), "Open Jupyter from IFLS project root."
os.chdir(root)
print("Project root:", root)
print("Outputs are saved locally and ignored by Git.")

## 1. Treatment data and payment years

Inspect BLSM as a distinct program in IFLS5 KSR1. Missing and ambiguous answers are not untreated households. Receipt in late 2014 may be different from the 2013 intervention.

In [ ]:
import runpy
runpy.run_path("02_scripts/10_blsm_2013/python/01_treatment_audit.py", run_name="__main__")
out = root / "05_outputs/tables/blsm_2013_python"
display(pd.read_csv(out / "01_blsm_timing_categories.csv"))

## 2. IFLS4–IFLS5 links

Use PIDLINK for people, but separately inspect tracking records for household split-offs. A household ID cannot be treated as a time-invariant person identifier.

In [ ]:
runpy.run_path("02_scripts/10_blsm_2013/python/02_panel_audit.py", run_name="__main__")
display(pd.read_csv(out / "02_roster_linkage_counts.csv"))

## 3. Household consumption components

We need to harmonize reporting periods (weekly/monthly/annual), treatment-related spending, household size, and price deflators before defining real consumption per capita.

In [ ]:
runpy.run_path("02_scripts/10_blsm_2013/python/03_welfare_inventory.py", run_name="__main__")
display(pd.read_csv(out / "03_welfare_file_inventory.csv"))

## 4. Interview dates and exposure timing

The interview dates must be checked before coding 2013 beneficiaries versus later assistance recipients.

In [ ]:
runpy.run_path("02_scripts/10_blsm_2013/python/04_interview_calendar_audit.py", run_name="__main__")
candidates = out / "04_interview_variable_candidates.csv"
if candidates.is_file():
    display(pd.read_csv(candidates))
else:
    print("No date candidates detected; consult IFLS questionnaires and cover files.")

## 5. Research decisions before ATT estimation

1. Which IFLS5 receipt code and first/last dates verify BLSM 2013?
2. Are there enough observed recipients and true nonrecipients after panel linkage?
3. What is our defensible observation unit when 2007 households split by 2014?
4. Which KS line items make up comparable consumption aggregates?
5. How will we account for inflation, household size and exposure to other programs?
6. Is the conditional parallel-trends assumption plausible with these two waves?
7. What baseline subgroups and sample counts are credible?

**Next stage (not implemented here):** balance/overlap checks, doubly robust panel DiD ATT, subgroup ATT, and a separate RAPSEM mediation feasibility analysis. RAPSEM's reference implementation is in R and cannot automatically estimate ATT.